# EHR Data Quality Audit
**Author:** Ragavi Rai | Healthcare Data Analyst

This notebook audits a synthetic EHR dataset for common data quality issues:
- Missing critical fields
- Duplicate patient records
- Invalid ICD-10 code formatting
- Data quality scoring per record

In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt

df = pd.read_csv('sample_ehr_data.csv')
print('Dataset loaded successfully')
print(f'Total records: {len(df)}')
df.head()

## 1. Missing Value Analysis

In [ ]:
critical_fields = ['chief_complaint', 'diagnosis', 'icd10_code', 'provider']

missing = df[critical_fields].isnull() | (df[critical_fields] == '')
missing_summary = missing.sum().reset_index()
missing_summary.columns = ['Field', 'Missing Count']
missing_summary['Missing %'] = (missing_summary['Missing Count'] / len(df) * 100).round(1)

print('=== Missing Value Report ===')
print(missing_summary.to_string(index=False))

plt.figure(figsize=(8, 4))
plt.bar(missing_summary['Field'], missing_summary['Missing %'], color='#E87C5A')
plt.title('Missing Values by Critical Field (%)')
plt.ylabel('Missing %')
plt.xlabel('Field')
plt.tight_layout()
plt.savefig('missing_values.png')
plt.show()

## 2. Duplicate Record Detection

In [ ]:
duplicates = df[df.duplicated(subset=['first_name','last_name','dob','visit_date'], keep=False)]
print(f'Duplicate records found: {len(duplicates)}')
print(duplicates[['patient_id','first_name','last_name','dob','visit_date']])

## 3. ICD-10 Code Validation

In [ ]:
def validate_icd10(code):
    if pd.isnull(code) or code == '':
        return False
    pattern = r'^[A-Z][0-9]{2}(\.?[0-9A-Z]{0,4})?$'
    return bool(re.match(pattern, str(code).strip()))

df['icd10_valid'] = df['icd10_code'].apply(validate_icd10)
invalid_codes = df[~df['icd10_valid']]
print(f'Invalid ICD-10 codes found: {len(invalid_codes)}')
print(invalid_codes[['patient_id','diagnosis','icd10_code']])

## 4. Data Quality Scoring

In [ ]:
def quality_score(row):
    score = 100
    if pd.isnull(row['chief_complaint']) or row['chief_complaint'] == '':
        score -= 25
    if pd.isnull(row['diagnosis']) or row['diagnosis'] == '':
        score -= 25
    if pd.isnull(row['provider']) or row['provider'] == '':
        score -= 25
    if not row['icd10_valid']:
        score -= 25
    return score

df['quality_score'] = df.apply(quality_score, axis=1)
print(f'Average quality score: {df["quality_score"].mean():.1f}/100')
print(f'Records scoring below 75: {len(df[df["quality_score"] < 75])}')

plt.figure(figsize=(8, 4))
plt.hist(df['quality_score'], bins=5, color='#4A90D9', edgecolor='white')
plt.title('Distribution of Data Quality Scores')
plt.xlabel('Quality Score')
plt.ylabel('Number of Records')
plt.tight_layout()
plt.savefig('quality_scores.png')
plt.show()

## 5. Export Audit Report

In [ ]:
audit_report = df[['patient_id','first_name','last_name','diagnosis',
                    'icd10_code','icd10_valid','quality_score']].copy()
audit_report['flag'] = audit_report['quality_score'].apply(
    lambda x: 'NEEDS REVIEW' if x < 100 else 'PASS'
)
audit_report.to_csv('audit_report.csv', index=False)
print('Audit report exported successfully')
print(audit_report.to_string(index=False))